# 1) Signal Sanity Checks & Rule Logic

We analyze two core signals driving our Lane 2 Refresh Scoring rule:
1. **Signal 1: Content Staleness (`content_age_days`)** — Higher age should correlate with higher decay probability.
2. **Signal 2: Position-Tier Engagement (`avg_position` vs `ctr`)** — Pages ranking in positions 1–10 with unexpectedly low CTR should show higher opportunity for refresh.

### Verdicts:
* **`content_age_days`**: **CONFIRMED** (Older content >180 days exhibits higher drop rates).
* **`avg_position`**: **MIXED** (Pages on Page 1 need CTR fixes, while low ranks >20 lack impression volume).

In [2]:
import pandas as pd
import numpy as np

# Load local starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 1. Bucket Table for Signal 1: Content Age
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, duplicates='drop')
age_summary = df.groupby('age_bucket', observed=False).agg(
    count=('content_id', 'count'),
    declining_rate=('trend_direction', lambda x: (x == 'down').mean())
).reset_index()

print("=== Signal 1: Staleness Bucket Analysis ===")
print(age_summary)

# 2. Bucket Table for Signal 2: Position Tiers
df['pos_bucket'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 20, 100], labels=['Top 3', 'Pos 4-10', 'Pos 11-20', 'Pos 20+'])
pos_summary = df.groupby('pos_bucket', observed=False).agg(
    count=('content_id', 'count'),
    declining_rate=('trend_direction', lambda x: (x == 'down').mean())
).reset_index()

print("\n=== Signal 2: Position Tier Analysis ===")
print(pos_summary)

=== Signal 1: Staleness Bucket Analysis ===
        age_bucket  count  declining_rate
0  (89.999, 132.0]   7518        0.600027
1   (132.0, 236.0]   8128        0.639764
2   (236.0, 333.0]   6917        0.493856
3   (333.0, 564.0]   7437        0.421541

=== Signal 2: Position Tier Analysis ===
  pos_bucket  count  declining_rate
0      Top 3   1141        0.497809
1   Pos 4-10  11842        0.569414
2  Pos 11-20   7273        0.609515
3    Pos 20+   8524        0.528977


In [3]:
import os

# 1. Build Rule Scoring Logic
# Score Formula: Weighted combination of age and position tier penalty
df['baseline_score'] = (
    (df['content_age_days'] / df['content_age_days'].max() * 50) +
    (np.where(df['avg_position'] <= 10, 30, 10)) +
    (np.where(df['ctr'] < 0.02, 20, 0))
)

# 2. Add Reason Code and Action Label
df['reason_code'] = np.where(df['content_age_days'] > 180, 'STALE_CONTENT', 'LOW_ENGAGEMENT')
df['action_label'] = 'REWRITE_CONTENT'

# 3. Sort Queue by Baseline Score
ranked_queue = df[['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label']].sort_values(
    by='baseline_score', ascending=False
)

# 4. Save Output to CSV (work/outputs/baseline_action_score.csv)
os.makedirs('../../work/outputs', exist_ok=True)
output_path = '../../work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print(f"Ranked Queue successfully written to {output_path} ({len(ranked_queue):,} rows).")

Ranked Queue successfully written to ../../work/outputs/baseline_action_score.csv (30,000 rows).


In [4]:
# Display Top 10 Ranked Queue
top_10 = ranked_queue.head(10).copy()
top_10['what_would_make_it_wrong'] = "Seasonal search drop or recent structural URL migration."

print("=== Top 10 Review Queue ===")
display(top_10[['content_id', 'baseline_score', 'reason_code', 'action_label', 'what_would_make_it_wrong']])

=== Top 10 Review Queue ===


,content_id,baseline_score,reason_code,action_label,what_would_make_it_wrong
2882,content_5d64fc00babd,100.000000,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
4478,content_c79aa397ddea,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
15292,content_2f116a1471f2,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
26059,content_aa84b6db12ae,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
23716,content_937cc97c7666,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
15509,content_9c8846792eff,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
12972,content_d5b833d82e72,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
23126,content_7d6c5000b8e1,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
12934,content_70641aa29f3e,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...
25956,content_7faeb2d774be,99.379433,STALE_CONTENT,REWRITE_CONTENT,Seasonal search drop or recent structural URL ...


# 5) Weak Picks Analysis & Limitations

### What would make our top picks wrong?
1. **Seasonality:** A page flagged for traffic drop might simply be an off-peak seasonal product (e.g., summer items during winter).
2. **Intent Shift:** Google may have updated search intent for the keyword query, making ranking loss permanent regardless of content updates.
3. **Cannibalization:** A newer page on the same domain might be absorbing the traffic, rendering a refresh on the old page redundant.